In [39]:
!uv pip install -q pandas plotly
!uv pip install -q kaleido nbformat

In [61]:
import pandas as pd
import glob
import json
import plotly.express as px

def load_data(paths):
    files = []
    for p in paths:
        files.extend(glob.glob(f"{p}/**/result.json", recursive=True))
    return pd.DataFrame([json.load(open(f)) for f in files])

paths = [
    './multirun/2026-04-26/full-17-15-01'
]
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=['times', 'autorange_min_run_time'])
df = df.sort_values(by=['model_id'], ascending=True, key=lambda col: col.str.lower())
df

,k,model_id,ef,batch_size,is_ref,mean,median,p25,p75,iqr,num_threads
28,50,google/gemma-3-1b-it,100,1,False,0.002078,0.002056,0.002045,0.002090,0.000044,6
29,50,google/gemma-3-1b-it,100,1,True,0.073391,0.071141,0.069177,0.075350,0.006173,6
30,50,google/gemma-3-1b-it,100,2,False,0.003259,0.003153,0.003137,0.003374,0.000237,6
31,50,google/gemma-3-1b-it,100,2,True,0.056254,0.055391,0.055316,0.055566,0.000249,6
32,50,google/gemma-3-1b-it,100,4,False,0.005008,0.005044,0.004785,0.005068,0.000283,6
...,...,...,...,...,...,...,...,...,...,...,...
107,50,Qwen/Qwen3-1.7B,200,16,True,0.062435,0.061526,0.060195,0.063071,0.002876,6
108,50,Qwen/Qwen3-1.7B,200,32,False,0.144343,0.141667,0.136610,0.149035,0.012425,6
109,50,Qwen/Qwen3-1.7B,200,32,True,0.077424,0.077000,0.075833,0.078197,0.002365,6
110,50,Qwen/Qwen3-1.7B,200,64,False,0.308238,0.309273,0.306096,0.312121,0.006024,6


In [62]:
index_cols = ['model_id', 'ef', 'batch_size']
ref_median = df[df['is_ref'] == True].set_index(index_cols)[['median']]
opt_median = df[df['is_ref'] == False].set_index(index_cols)[['median']]

speedup = ref_median / opt_median

dfs = ref_median.join(opt_median, lsuffix='_ref', rsuffix='_nonref')
dfs['speedup'] = speedup

dfs = dfs.reset_index()

dfs[dfs["batch_size"] == 1]

,model_id,ef,batch_size,median_ref,median_nonref,speedup
0,google/gemma-3-1b-it,100,1,0.071141,0.002056,34.601654
7,google/gemma-3-1b-it,200,1,0.069897,0.003742,18.679049
14,google/gemma-3-270m-it,100,1,0.038932,0.001183,32.909552
21,google/gemma-3-270m-it,200,1,0.041130,0.001783,23.067863
28,LiquidAI/LFM2.5-1.2B-Instruct,100,1,0.031777,0.002819,11.272437
35,LiquidAI/LFM2.5-1.2B-Instruct,200,1,0.031659,0.004640,6.823060
42,Qwen/Qwen3-0.6B,100,1,0.035793,0.002114,16.931410
49,Qwen/Qwen3-0.6B,200,1,0.036060,0.003842,9.385737
56,Qwen/Qwen3-1.7B,100,1,0.072527,0.003617,20.051700
63,Qwen/Qwen3-1.7B,200,1,0.072435,0.006900,10.497826


In [63]:
paper_style_dict = {
    'layout.plot_bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.font.family': 'monospace', # 'Times New Roman',
    'layout.font.size': 20,
    'layout.xaxis.linecolor': 'black',
    'layout.xaxis.ticks': 'outside',
    'layout.xaxis.mirror': True,
    'layout.xaxis.showline': True,
    'layout.xaxis.showgrid': True,
    'layout.xaxis.gridcolor': 'lightgray',
    'layout.xaxis.gridwidth': 0.1,
    'layout.yaxis.linecolor': 'black',
    'layout.yaxis.ticks': 'outside',
    'layout.yaxis.mirror': True,
    'layout.yaxis.showline': True,
    'layout.yaxis.showgrid': True,
    'layout.yaxis.gridcolor': 'lightgray',
    'layout.yaxis.gridwidth': 0.1,
    'layout.autosize': False,
    'layout.showlegend': True,
    'layout.legend.bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.legend.xanchor': 'right',
    'layout.legend.x': 1,
    'layout.legend.font.family': 'monospace',
    'layout.legend.font.size': 16,
    'layout.xaxis.gridwidth': 1, 
    'layout.yaxis.gridwidth': 1,
    'layout.margin': {'l': 20, 'r': 20, 't': 20, 'b': 20},
}

In [ ]:
def plot(df, ylim_max=None):
    fig = px.line(
        df, 
        x="batch_size", 
        y="speedup",
        range_y=[0, ylim_max],
        range_x=[1, 32],
        color="model_id",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Set1,
        labels={
            "batch_size": "Batch Size",
            "speedup": "Speedup (×)",
            "model_id": "Model Architecture"
        }
    )

    fig.add_hline(
        y=1, 
        line_dash="dash", 
        line_color="black", 
        annotation_text="Baseline", 
        annotation_position="top left",
        line_width=1
    )

    fig.update_layout(xaxis = dict(tickvals = list(df["batch_size"])))

    fig.update(**paper_style_dict)
    return fig

dfs = dfs[(dfs["model_id"] != "LiquidAI/LFM2.5-1.2B-Instruct") & (dfs["batch_size"] <= 32)]

print("EF 100")
fig = plot(dfs[(dfs["ef"] == 100)], ylim_max=35)
# fig.write_image("head-speedup-ef-100.png", scale=4, width=800, height=500)
fig.write_image("head-speedup-ef-100.pdf")
fig.show()

print("EF 200")
fig = plot(dfs[(dfs["ef"] == 200)], ylim_max=25)
# fig.write_image("head-speedup-ef-200.png", scale=4, width=800, height=500)
fig.write_image("head-speedup-ef-200.pdf")
fig.show()

EF 100


EF 200
